In [1]:
"""
prepare_dataset.py
------------------
Downloads, cleans, filters and formats 3 data sources into a single
ChatML-format JSONL file ready for QLoRA SFT training.

Sources used:
  1. bigcode/self-oss-instruct-sc2-exec-filter-50k  (execution-verified code gen)
  2. thudm/DebugBench                               (multi-language debugging)
  3. sahil2801/CodeAlpaca-20k                       (instruction-style code tasks)

Output: data/train.jsonl, data/eval.jsonl, data/stats.json
before this install this pip install datasets transformers trl peft bitsandbytes accelerate
"""

import json
import random
import re
import hashlib
from pathlib import Path
from collections import defaultdict
from datasets import load_dataset
from tqdm import tqdm

# ── Config ──────────────────────────────────────────────────────────────────
SEED              = 42
TARGET_TRAIN      = 40_000
TARGET_EVAL       = 2_000
MIN_RESPONSE_LEN  = 80      # chars — filters "yes" / one-liner trash answers
MAX_SEQ_CHARS     = 6_000   # avoid extremely long samples that hurt training
DEDUP_THRESHOLD   = 0.85    # jaccard similarity ceiling before dropping
BAD_PATTERNS      = [
    r"TODO",
    r"raise NotImplementedError",
    r"pass\s*#.*implement",
    r"Your code here",
    r"# implement this",
    r"\.{3}\s*$",            # trailing ellipsis
]

SYSTEM_PROMPT = (
    "You are Forge, a precision coding assistant. "
    "When writing code: produce correct, clean, well-named solutions first, "
    "then briefly explain key decisions, then note edge cases. "
    "When debugging: state the root cause in one sentence, show the fixed code, "
    "explain why the bug occurred. "
    "Never write placeholder code. Never use TODO. Always complete what you start."
)

random.seed(SEED)
out_dir = Path(__file__).parent
out_dir.mkdir(exist_ok=True)

# ── Helpers ──────────────────────────────────────────────────────────────────
_compiled_bad = [re.compile(p, re.IGNORECASE) for p in BAD_PATTERNS]

def is_clean(text: str) -> bool:
    """Return True if text passes all quality filters."""
    if len(text) < MIN_RESPONSE_LEN:
        return False
    if len(text) > MAX_SEQ_CHARS:
        return False
    for pattern in _compiled_bad:
        if pattern.search(text):
            return False
    return True

def shingle(text: str, k: int = 4) -> set:
    """Character k-shingles for Jaccard dedup."""
    text = text.lower().strip()
    return {text[i:i+k] for i in range(len(text) - k + 1)}

def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    inter = len(a & b)
    return inter / (len(a) + len(b) - inter)

def to_chatml(user: str, assistant: str) -> dict:
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user.strip()},
            {"role": "assistant", "content": assistant.strip()},
        ]
    }

def fingerprint(text: str) -> str:
    return hashlib.md5(text.lower().strip().encode()).hexdigest()

# ── Source 1: self-oss-instruct (execution-verified) ─────────────────────────
def load_self_oss(limit: int = 20_000) -> list[dict]:
    print("\n[1/3] Loading self-oss-instruct-sc2-exec-filter-50k ...")
    ds = load_dataset(
        "bigcode/self-oss-instruct-sc2-exec-filter-50k",
        split="train"
    )
    samples = []
    for row in tqdm(ds, desc="  self-oss"):
        prompt   = row.get("prompt", "") or row.get("instruction", "")
        response = row.get("response", "") or row.get("output", "")
        if not prompt or not response:
            continue
        if not is_clean(response):
            continue
        samples.append(to_chatml(prompt, response))
        if len(samples) >= limit:
            break
    print(f"  → kept {len(samples):,} samples")
    return samples

# ── Source 2: DebugBench ──────────────────────────────────────────────────────
def load_debugbench(limit: int = 10_000) -> list[dict]:
    print("\n[2/3] Loading DebugBench ...")
    ds = load_dataset("Rtian/DebugBench", split="test") # only split available
    samples = []
    for row in tqdm(ds, desc="  DebugBench"):
        buggy_code  = row.get("buggy_code", "")
        fixed_code  = row.get("fixed_code", "")
        bug_type    = row.get("bug_type", "bug")
        language    = row.get("language", "code")
        
        
        # STRICT LANGUAGE FILTER
        allowed_langs = ["python", "c", "cpp", "c++","java", "javascript", "js","sql"]
        if language.lower() not in allowed_langs:
            continue

        description = row.get("description", "")

        if not buggy_code or not fixed_code:
            continue

        user = (
            f"The following {language} code has a bug ({bug_type}).\n"
            f"{'Description: ' + description + chr(10) if description else ''}"
            f"Find and fix it.\n\n```{language.lower()}\n{buggy_code}\n```"
        )
        assistant = (
            f"**Root Cause:** {bug_type} in the code above.\n\n"
            f"**Fixed Code:**\n```{language.lower()}\n{fixed_code}\n```\n\n"
            f"**Explanation:** The bug was corrected by addressing the {bug_type} issue. "
            f"Always verify boundary conditions and type assumptions in {language} code."
        )
        if not is_clean(fixed_code):
            continue
        samples.append(to_chatml(user, assistant))
        if len(samples) >= limit:
            break
    print(f"  → kept {len(samples):,} samples")
    return samples

# ── Source 3: CodeAlpaca ──────────────────────────────────────────────────────
def load_codealpaca(limit: int = 15_000) -> list[dict]:
    print("\n[3/3] Loading CodeAlpaca-20k ...")
    ds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    samples = []
    for row in tqdm(ds, desc="  CodeAlpaca"):
        instruction = row.get("instruction", "")
        output      = row.get("output", "")
        context     = row.get("input", "")
        if context:
            user = f"{instruction}\n\nContext:\n```\n{context}\n```"
        else:
            user = instruction
        if not instruction or not output:
            continue
        if not is_clean(output):
            continue
        samples.append(to_chatml(user, output))
        if len(samples) >= limit:
            break
    print(f"  → kept {len(samples):,} samples")
    return samples

# ── Deduplication ─────────────────────────────────────────────────────────────
def deduplicate(samples: list[dict]) -> list[dict]:
    print("\n[Dedup] Running deduplication ...")
    seen_fps  = set()
    seen_shingles = []
    kept = []

    for s in tqdm(samples, desc="  dedup"):
        resp = s["messages"][2]["content"]
        fp   = fingerprint(resp)
        if fp in seen_fps:
            continue
        sh = shingle(resp)
        # Only compare against last 2000 to keep it O(n) reasonable
        too_similar = any(
            jaccard(sh, prev) > DEDUP_THRESHOLD
            for prev in seen_shingles[-2000:]
        )
        if too_similar:
            continue
        seen_fps.add(fp)
        seen_shingles.append(sh)
        kept.append(s)

    print(f"  → {len(samples):,} → {len(kept):,} after dedup")
    return kept

# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    all_samples = []
    all_samples.extend(load_self_oss(limit=20_000))
    all_samples.extend(load_debugbench(limit=10_000))
    all_samples.extend(load_codealpaca(limit=15_000))

    print(f"\n[Total] Raw samples: {len(all_samples):,}")

    # Deduplicate
    all_samples = deduplicate(all_samples)

    # Shuffle
    random.shuffle(all_samples)

    # Split
    eval_samples  = all_samples[:TARGET_EVAL]
    train_samples = all_samples[TARGET_EVAL:TARGET_EVAL + TARGET_TRAIN]

    print(f"\n[Split] Train: {len(train_samples):,}  |  Eval: {len(eval_samples):,}")

    # Write JSONL
    train_path = out_dir / "train.jsonl"
    eval_path  = out_dir / "eval.jsonl"

    with open(train_path, "w") as f:
        for s in train_samples:
            f.write(json.dumps(s) + "\n")

    with open(eval_path, "w") as f:
        for s in eval_samples:
            f.write(json.dumps(s) + "\n")

    # Stats
    stats = {
        "train_samples": len(train_samples),
        "eval_samples":  len(eval_samples),
        "total_kept":    len(all_samples),
        "min_response_len": MIN_RESPONSE_LEN,
        "max_seq_chars":    MAX_SEQ_CHARS,
        "system_prompt":    SYSTEM_PROMPT,
    }
    with open(out_dir / "stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    print(f"\n✓ train.jsonl  → {train_path}")
    print(f"✓ eval.jsonl   → {eval_path}")
    print(f"✓ stats.json   → {out_dir / 'stats.json'}")
    print("\nDataset ready for training.")

if __name__ == "__main__":
    main()



[1/3] Loading self-oss-instruct-sc2-exec-filter-50k ...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/90.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50661 [00:00<?, ? examples/s]

  self-oss:  40%|███▉      | 20015/50661 [00:01<00:02, 10780.98it/s]


...

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

  CodeAlpaca: 100%|██████████| 20022/20022 [00:00<00:00, 34476.51it/s]


  → kept 13,141 samples

[Total] Raw samples: 33,141

[Dedup] Running deduplication ...


  dedup: 100%|██████████| 33141/33141 [10:20<00:00, 53.38it/s] 


  → 33,141 → 33,074 after dedup

[Split] Train: 31,074  |  Eval: 2,000

✓ train.jsonl  → /home/jovyan/train.jsonl
✓ eval.jsonl   → /home/jovyan/eval.jsonl
✓ stats.json   → /home/jovyan/stats.json

Dataset ready for training.


## Traning

In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import time
from pathlib import Path
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)
from trl import SFTConfig, SFTTrainer

# ── Config ──
MODEL_ID      = "google/gemma-3-27b-it"  # <--- STABLE MODEL
DATA_DIR      = Path("data")
OUTPUT_DIR    = Path("output/gemma3-forge-v1")

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear", # <--- CLEAN TARGETS
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
)

TRAIN_CONFIG = dict(
    max_steps=1000,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    warmup_ratio=0.03,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=250,
)

class TrainingLogger(TrainerCallback):
    def __init__(self):
        self.start_time = time.time()
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"Step {state.global_step} | Loss: {logs['loss']:.4f}")

def make_formatter(tokenizer):
    def format_sample(example):
        messages = [dict(m) for m in example["messages"]]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        return {"text": text}
    return format_sample

def main():
    print("Starting Stable Gemma 3 QLoRA...")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        quantization_config=BNB_CONFIG,
        device_map={"": 0},
        attn_implementation="flash_attention_2",   
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LORA_CONFIG)

    train_ds = load_dataset("json", data_files=str(DATA_DIR / "train.jsonl"), split="train")
    formatter = make_formatter(tokenizer)
    train_ds = train_ds.map(formatter, remove_columns=["messages"])

    sft_config = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        dataset_text_field="text",
        max_length=2048,
        packing=False,
        **TRAIN_CONFIG,
    )
    
    # --- GEMMA 3 MULTIMODAL PATCH ---
    # Forces the model to treat all incoming data as text tokens (0)
    original_gemma_forward = model.__class__.forward

    def patched_forward(self, *args, **kwargs):
        if kwargs.get("token_type_ids") is None and "input_ids" in kwargs:
            kwargs["token_type_ids"] = torch.zeros_like(kwargs["input_ids"])
        return original_gemma_forward(self, *args, **kwargs)

    model.__class__.forward = patched_forward
    # --------------------------------
    
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        processing_class=tokenizer,
        callbacks=[TrainingLogger()],
    )

    trainer.train()
    trainer.save_model(str(OUTPUT_DIR / "final"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "final"))

if __name__ == "__main__":
    main()

Starting Stable Gemma 3 QLoRA...


Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.
Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss
10,1.827733
20,0.987651
30,0.577356
40,0.328064
50,0.220568
60,0.209774
70,0.205806
80,0.205899
90,0.209552
100,0.201399


Step 10 | Loss: 1.8277
Step 20 | Loss: 0.9877
Step 30 | Loss: 0.5774
Step 40 | Loss: 0.3281
Step 50 | Loss: 0.2206
Step 60 | Loss: 0.2098
Step 70 | Loss: 0.2058
Step 80 | Loss: 0.2059
Step 90 | Loss: 0.2096
Step 100 | Loss: 0.2014
Step 110 | Loss: 0.2082
Step 120 | Loss: 0.1915
Step 130 | Loss: 0.2135
Step 140 | Loss: 0.2051
Step 150 | Loss: 0.1956
Step 160 | Loss: 0.1955
Step 170 | Loss: 0.1917
Step 180 | Loss: 0.2111
Step 190 | Loss: 0.2040
Step 200 | Loss: 0.1924
Step 210 | Loss: 0.2112
Step 220 | Loss: 0.1931
Step 230 | Loss: 0.1972
Step 240 | Loss: 0.1905
Step 250 | Loss: 0.1846
Step 260 | Loss: 0.2074
Step 270 | Loss: 0.2180
Step 280 | Loss: 0.2180
Step 290 | Loss: 0.1923
Step 300 | Loss: 0.1884
Step 310 | Loss: 0.2214
Step 320 | Loss: 0.1822
Step 330 | Loss: 0.2014
Step 340 | Loss: 0.2059
Step 350 | Loss: 0.2043
Step 360 | Loss: 0.1983
Step 370 | Loss: 0.1879
Step 380 | Loss: 0.2158
Step 390 | Loss: 0.2100
Step 400 | Loss: 0.2029
Step 410 | Loss: 0.1926
Step 420 | Loss: 0.2087
S

In [1]:
# Cell 0 — Run once per session before merge_and_export
import subprocess, sys

# 1. Force downgrade Triton to the last stable version with the 'ops' module
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "triton==3.1.0"], check=True)

# 2. Upgrade the rest of the stack
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "peft"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers>=4.49.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "huggingface_hub"], check=True)

print("✓ All dependencies installed and Triton patched!")

✓ All dependencies installed and Triton patched!


In [2]:
!pip install --upgrade "torchao>=0.16.0"

  Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (20 kB)
Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (3.2 MB)
  Attempting uninstall: torchao
    Found existing installation: torchao 0.11.0+git
    Uninstalling torchao-0.11.0+git:
      Successfully uninstalled torchao-0.11.0+git


In [3]:
!pip install sentencepiece

  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (1.4 MB)


## Merge and Export

In [2]:
"""
merge_and_export.py
───────────────────
Permanent fix edition — no tokenizer hotpatch needed.

Steps before running this:
  1. Clone a FRESH llama.cpp (b3447 or later):
       git clone https://github.com/ggerganov/llama.cpp
       cd llama.cpp
       make -j$(nproc) GGML_CUDA=1
       export LLAMA_CPP_PATH=$(pwd)

  2. Do NOT patch convert_hf_to_gguf.py — the new version recognises
     Gemma 3's SentencePiece tokenizer natively without the llama-bpe hotpatch.

  3. Run:
       python merge_and_export.py

  4. Download output/gemma3-forge-Q4_K_M.gguf to your RTX 3060 machine.
     Terminate the cloud instance after download to stop billing.
"""

import os
import subprocess
import sys
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT           = Path(__file__).parent
BASE_MODEL_ID  = "google/gemma-3-27b-it"
ADAPTER_PATH   = ROOT / "output" / "gemma3-forge-v1" / "final"
MERGED_PATH    = ROOT / "output" / "gemma3-forge-v1" / "merged"
INTERIM_GGUF   = ROOT / "output" / "interim-f16.gguf"
GGUF_OUT       = ROOT / "output" / "gemma3-forge-Q4_K_M.gguf"
LLAMA_CPP_PATH = Path(os.getenv("LLAMA_CPP_PATH", str(ROOT / "llama.cpp")))


# ── Step 1: Merge ─────────────────────────────────────────────────────────────
def merge():
    print("\n[Merge 1/3] Loading base model (bfloat16, CPU) ...")
    # device_map="cpu" is intentional — loading 27B to GPU for merge needs ~54 GB VRAM.
    # CPU is slower but safe on any machine.
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="cpu",
        low_cpu_mem_usage=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_PATH))

    print("[Merge 2/3] Merging LoRA adapter into base weights ...")
    model = PeftModel.from_pretrained(
        base,
        str(ADAPTER_PATH),
        autocast_adapter_dtype=False,
    )
    model = model.merge_and_unload()
    model.eval()

    print(f"[Merge 3/3] Saving merged model → {MERGED_PATH}")
    MERGED_PATH.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(MERGED_PATH), safe_serialization=True)
    tokenizer.save_pretrained(str(MERGED_PATH))
    print("✓ Merge complete\n")

    _verify_merged()


def _verify_merged():
    """Abort early if the merged model looks incomplete or corrupt."""
    if not (MERGED_PATH / "config.json").exists():
        print("✗ config.json missing — merge failed or was interrupted.")
        sys.exit(1)

    shards    = list(MERGED_PATH.glob("*.safetensors"))
    total_gb  = sum(p.stat().st_size for p in shards) / 1e9

    print(f"  Merged model: {len(shards)} shard(s), {total_gb:.1f} GB")

    # Gemma 3 27B bfloat16 should be ~54 GB
    if total_gb < 10:
        print(f"✗ Merged model is {total_gb:.1f} GB — way too small. Aborting.")
        sys.exit(1)

    print("  ✓ Merged model looks correct\n")


# ── Step 2: Export GGUF ───────────────────────────────────────────────────────
def export_gguf():
    """
    Two-step GGUF export:
      1. convert_hf_to_gguf.py → lossless f16 GGUF
      2. llama-quantize         → Q4_K_M (~15 GB)

    KEY DIFFERENCE from old approach:
      Using a FRESH llama.cpp (b3447+) means convert_hf_to_gguf.py auto-detects
      Gemma 3's SentencePiece tokenizer WITHOUT the llama-bpe hotpatch.
      The hotpatch was what caused "distances", "neighbor", "weight" to be
      silently dropped from generated code.
    """
    convert_script = LLAMA_CPP_PATH / "convert_hf_to_gguf.py"

    # Locate llama-quantize — path differs between make and cmake builds
    quantize_candidates = [
        LLAMA_CPP_PATH / "llama-quantize",                    # make build (most common)
        LLAMA_CPP_PATH / "build" / "bin" / "llama-quantize",  # cmake build
        LLAMA_CPP_PATH / "build" / "bin" / "quantize",        # cmake old name
    ]
    quantize_bin = next((p for p in quantize_candidates if p.exists()), None)

    # Fail loudly with actionable instructions
    if not convert_script.exists():
        print(f"✗ convert_hf_to_gguf.py not found at {convert_script}")
        print("  Clone a fresh llama.cpp and build it:")
        print("    git clone https://github.com/ggerganov/llama.cpp")
        print("    cd llama.cpp && make -j$(nproc) GGML_CUDA=1")
        print("    export LLAMA_CPP_PATH=$(pwd)")
        sys.exit(1)

    if quantize_bin is None:
        print(f"✗ llama-quantize binary not found in {LLAMA_CPP_PATH}")
        print("  Run: make -j$(nproc) GGML_CUDA=1  inside llama.cpp/")
        sys.exit(1)

    print(f"  convert : {convert_script.name}")
    print(f"  quantize: {quantize_bin}\n")

    # ── 2a: safetensors → f16 GGUF ───────────────────────────────────────────
    print("[GGUF 1/2] Converting to f16 GGUF (10–20 min, reads ~54 GB) ...")
    result = subprocess.run([
        sys.executable, str(convert_script),
        str(MERGED_PATH),
        "--outfile", str(INTERIM_GGUF),
        "--outtype", "f16",
        # No --vocab-type flag — fresh llama.cpp auto-detects Gemma 3's tokenizer.
        # DO NOT add the llama-bpe hotpatch here.
    ])

    if result.returncode != 0:
        print("✗ f16 conversion failed.")
        print()
        print("  If the error says 'BPE pre-tokenizer was not recognized':")
        print("  → Your llama.cpp is too old. Clone a fresh copy:")
        print("      git clone https://github.com/ggerganov/llama.cpp")
        print("  → Then set: export LLAMA_CPP_PATH=/path/to/new/llama.cpp")
        print("  → Do NOT patch convert_hf_to_gguf.py this time.")
        sys.exit(1)

    interim_gb = INTERIM_GGUF.stat().st_size / 1e9
    print(f"  ✓ f16 GGUF ready: {interim_gb:.1f} GB\n")

    # ── 2b: f16 GGUF → Q4_K_M ────────────────────────────────────────────────
    print("[GGUF 2/2] Quantising to Q4_K_M ...")
    result = subprocess.run([
        str(quantize_bin),
        str(INTERIM_GGUF),
        str(GGUF_OUT),
        "Q4_K_M",
    ])

    if result.returncode != 0:
        print("✗ Quantisation failed.")
        sys.exit(1)

    # Remove the large intermediate file
    if INTERIM_GGUF.exists():
        INTERIM_GGUF.unlink()
        print(f"  Removed interim file ({interim_gb:.1f} GB freed)")

    final_gb = GGUF_OUT.stat().st_size / 1e9
    print(f"\n✓ Done. Final GGUF: {final_gb:.1f} GB → {GGUF_OUT}")
    print()
    print("─" * 54)
    print("  NEXT STEPS")
    print("─" * 54)
    print(f"  1. Download the GGUF to your local machine:")
    print(f"       scp <user>@<cloud-ip>:{GGUF_OUT} ~/")
    print()
    print(f"  2. Start the server on your RTX 3060:")
    print(f"       python main.py --model ~/gemma3-forge-Q4_K_M.gguf")
    print()
    print(f"  3. ⚠  Terminate your cloud instance now to stop billing.")
    print("─" * 54)


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 54)
    print("  Forge — Merge + GGUF Export  (Permanent Fix)")
    print("=" * 54)
    print(f"  Base model  : {BASE_MODEL_ID}")
    print(f"  Adapter     : {ADAPTER_PATH}")
    print(f"  Merged path : {MERGED_PATH}")
    print(f"  GGUF output : {GGUF_OUT}")
    print(f"  llama.cpp   : {LLAMA_CPP_PATH}")
    print()

    #merge()
    export_gguf()

  Forge — Merge + GGUF Export  (Permanent Fix)
  Base model  : google/gemma-3-27b-it
  Adapter     : /home/jovyan/output/gemma3-forge-v1/final
  Merged path : /home/jovyan/output/gemma3-forge-v1/merged
  GGUF output : /home/jovyan/output/gemma3-forge-Q4_K_M.gguf
  llama.cpp   : /home/jovyan/llama.cpp

  convert : convert_hf_to_gguf.py
  quantize: /home/jovyan/llama.cpp/build/bin/llama-quantize

[GGUF 1/2] Converting to f16 GGUF (10–20 min, reads ~54 GB) ...


INFO:hf-to-gguf:Loading model: merged
INFO:hf-to-gguf:Model architecture: Gemma3ForConditionalGeneration
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00005-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00006-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00007-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00008-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00009-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00010-of-00012.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 

  ✓ f16 GGUF ready: 54.0 GB

[GGUF 2/2] Quantising to Q4_K_M ...


llama_print_build_info: build = 8840 (9e5647aff)
llama_print_build_info: built with GNU 13.3.0 for Linux x86_64
main: quantizing '/home/jovyan/output/interim-f16.gguf' to '/home/jovyan/output/gemma3-forge-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 36 key-value pairs and 808 tensors from /home/jovyan/output/interim-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 64
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                               general.name str              = Merged
llama_model_loader: - kv   5:       


main: quantize time = 321699.73 ms
main:    total time = 321699.73 ms
  Removed interim file (54.0 GB freed)

✓ Done. Final GGUF: 16.5 GB → /home/jovyan/output/gemma3-forge-Q4_K_M.gguf

──────────────────────────────────────────────────────
  NEXT STEPS
──────────────────────────────────────────────────────
  1. Download the GGUF to your local machine:
       scp <user>@<cloud-ip>:/home/jovyan/output/gemma3-forge-Q4_K_M.gguf ~/

  2. Start the server on your RTX 3060:
       python main.py --model ~/gemma3-forge-Q4_K_M.gguf

  3. ⚠  Terminate your cloud instance now to stop billing.
──────────────────────────────────────────────────────


In [4]:
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil

merged_dir = Path("output/gemma3-forge-v1/merged")
base_model = "google/gemma-3-27b-it"

print("1. Fetching pristine tokenizer from Google base model...")
vocab_dir = snapshot_download(
    repo_id=base_model,
    allow_patterns=["tokenizer*", "special_tokens_map.json"]
)

print("2. Overwriting corrupted LoRA tokenizer files...")
for file in Path(vocab_dir).glob("*"):
    if file.is_file():
        shutil.copy(file, merged_dir / file.name)
        print(f"   ✓ Restored {file.name}")

print("\n✓ Tokenizer hash fixed! llama.cpp will now recognize it perfectly.")

1. Fetching pristine tokenizer from Google base model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

2. Overwriting corrupted LoRA tokenizer files...
   ✓ Restored model-00003-of-00012.safetensors
   ✓ Restored config.json
   ✓ Restored added_tokens.json
   ✓ Restored model.safetensors.index.json
   ✓ Restored tokenizer_config.json
   ✓ Restored model-00002-of-00012.safetensors
   ✓ Restored model-00008-of-00012.safetensors
   ✓ Restored model-00005-of-00012.safetensors
   ✓ Restored model-00004-of-00012.safetensors
   ✓ Restored tokenizer.model
   ✓ Restored model-00006-of-00012.safetensors
   ✓ Restored model-00009-of-00012.safetensors
   ✓ Restored model-00010-of-00012.safetensors
   ✓ Restored tokenizer.json
   ✓ Restored model-00012-of-00012.safetensors
   ✓ Restored special_tokens_map.json
   ✓ Restored generation_config.json
   ✓ Restored model-00001-of-00012.safetensors
   ✓ Restored model-00011-of-00012.safetensors
   ✓ Restored model-00007-of-00012.safetensors

✓ Tokenizer hash fixed! llama.cpp will now recognize it perfectly.


In [1]:
# import os

# file_path = "llama.cpp/convert_hf_to_gguf.py"

# with open(file_path, "r") as f:
#     content = f.read()

# # This looks for the exact error line and replaces it with a command 
# # that forces the script to use the standard "llama-bpe" tokenizer logic
# error_line = 'raise NotImplementedError("BPE pre-tokenizer was not recognized - update get_vocab_base_pre()")'
# fix_line = 'return "llama-bpe"  # HOTPATCH FOR GEMMA 3 HASH'

# if error_line in content:
#     content = content.replace(error_line, fix_line)
#     with open(file_path, "w") as f:
#         f.write(content)
#     print("✓ Successfully hot-patched llama.cpp conversion script!")
# else:
#     print("Script is already patched or line not found.")

In [3]:
import json
from pathlib import Path
from safetensors.torch import load_file, save_file

merged_dir = Path("output/gemma3-forge-v1/merged")

print("1. Fixing prefixes in safetensors (this takes a couple minutes)...")
for file in merged_dir.glob("*.safetensors"):
    print(f"   Processing {file.name}...")
    
    # Load the 27GB shard into RAM
    state_dict = load_file(file)
    new_state_dict = {}
    
    # Strip the extra "model." prefix
    for k, v in state_dict.items():
        new_key = k.replace("model.model.", "model.")
        new_state_dict[new_key] = v
        
    # Overwrite the file with the corrected keys
    save_file(new_state_dict, file)
    print(f"   ✓ Saved {file.name}")

print("\n2. Updating the index map...")
index_path = merged_dir / "model.safetensors.index.json"
if index_path.exists():
    with open(index_path, "r") as f:
        index_data = json.load(f)
    
    new_weight_map = {}
    for k, v in index_data.get("weight_map", {}).items():
        new_key = k.replace("model.model.", "model.")
        new_weight_map[new_key] = v
        
    index_data["weight_map"] = new_weight_map
    with open(index_path, "w") as f:
        json.dump(index_data, f, indent=2)

print("\n✓ All done! Your model is now 100% llama.cpp compatible.")

1. Fixing prefixes in safetensors (this takes a couple minutes)...
   Processing model-00001-of-00002.safetensors...
   ✓ Saved model-00001-of-00002.safetensors
   Processing model-00002-of-00002.safetensors...
   ✓ Saved model-00002-of-00002.safetensors

2. Updating the index map...

✓ All done! Your model is now 100% llama.cpp compatible.


### *to remove the dead vision weights 

In [2]:
import json
from pathlib import Path
from safetensors.torch import load_file, save_file

merged_dir = Path("output/gemma3-forge-v1/merged")

print("1. Removing ALL vision weights from safetensors...")
for file in merged_dir.glob("*.safetensors"):
    print(f"   Processing {file.name}...")
    state_dict = load_file(file)
    
    # Keep only keys that DO NOT contain "multi_modal" AND DO NOT contain "vision"
    new_state_dict = {
        k: v for k, v in state_dict.items() 
        if "multi_modal" not in k and "vision" not in k
    }
    
    if len(new_state_dict) < len(state_dict):
        print(f"      Removed {len(state_dict) - len(new_state_dict)} vision tensors.")
        save_file(new_state_dict, file)
        print(f"   ✓ Saved {file.name}")
    else:
        print(f"   ✓ No vision tensors found in this shard.")

print("\n2. Updating the index map...")
index_path = merged_dir / "model.safetensors.index.json"
if index_path.exists():
    with open(index_path, "r") as f:
        index_data = json.load(f)
    
    # Filter the index map for both prefixes
    new_weight_map = {
        k: v for k, v in index_data.get("weight_map", {}).items() 
        if "multi_modal" not in k and "vision" not in k
    }
        
    index_data["weight_map"] = new_weight_map
    with open(index_path, "w") as f:
        json.dump(index_data, f, indent=2)

print("\n✓ All vision components (tower and projector) completely purged!")

1. Removing ALL vision weights from safetensors...
   Processing model-00001-of-00002.safetensors...
   ✓ No vision tensors found in this shard.
   Processing model-00002-of-00002.safetensors...
      Removed 439 vision tensors.
   ✓ Saved model-00002-of-00002.safetensors

2. Updating the index map...

✓ All vision components (tower and projector) completely purged!


In [4]:
!pip install requests tqdm click datasets

In [6]:
CMAKE_ARGS="-DGGML_CUDA=on" 

In [7]:
!pip install --upgrade --force-reinstall fastapi uvicorn pydantic "llama-cpp-python[server]"

  Using cached fastapi-0.136.0-py3-none-any.whl.metadata (28 kB)
  Using cached uvicorn-0.44.0-py3-none-any.whl.metadata (6.7 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 105.4 MB/s  0:00:00eta 0:00:01
  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached starlette-1.0.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
Using cached fastapi-0.136.